### Capturing Front Office Executive Rosters across teams, years and tenure from RealGM

In [1]:
import pandas as pd

url = "https://basketball.realgm.com/nba/staff-members/16/General-Manager/Historical?utm_source=chatgpt.com"

tables = pd.read_html(url)

len(tables)

HTTPError: HTTP Error 403: Forbidden

In [4]:
from pathlib import Path

print("Current directory:", Path.cwd())

raw_dir = Path("../data/raw")

print("Raw directory exists:", raw_dir.exists())

if raw_dir.exists():
    print("Folders inside raw:")
    for folder in raw_dir.iterdir():
        print(folder.name)

Current directory: /Users/jonathantran/Desktop/nba_economics/notebooks
Raw directory exists: True
Folders inside raw:
.DS_Store
salary_cap_apron
team_payroll
spotrac
contracts
other
CBA_regime
real_gm
front_office
Ownership
player_salaries
larry_coon
awards
hoops_hype
basketball_reference


In [5]:
from pathlib import Path

folder = Path("../data/raw/front_office")

for file in folder.iterdir():
    print(repr(file.name))

'Front Office Tenures .csv'


In [6]:
front_office = pd.read_csv("../data/raw/front_office/Front Office Tenures .csv")

print(front_office.shape)
display(front_office.head())
front_office.info()

(269, 7)


,Staff,Team,Start Season,End Season,Years In Role,Birth Date,Nationality
0,Otto Adams,DET,1956-1957,1956-1957,1,-,-
1,Koby Altman,CLE,2017-2018,2021-2022,5,-,United States
2,Jack Ankerson,SAN,1973-1974,1973-1974,1,-,-
3,Al Attles,"GOS, GOS",1976-1977,1985-1986,20,"Nov 7, 1936",United States
4,Red Auerbach,"ATL, BOS",1949-1950,1983-1984,35,"Sep 20, 1917",United States


<class 'pandas.DataFrame'>
RangeIndex: 269 entries, 0 to 268
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Staff          269 non-null    str  
 1   Team           269 non-null    str  
 2   Start Season   269 non-null    str  
 3   End Season     269 non-null    str  
 4   Years In Role  269 non-null    int64
 5   Birth Date     269 non-null    str  
 6   Nationality    269 non-null    str  
dtypes: int64(1), str(6)
memory usage: 14.8 KB


In [7]:
# Create a copy so the raw DataFrame remains unchanged
gm = front_office.copy()

# Keep only the columns needed
gm = gm[
    ["Staff", "Team", "Start Season", "End Season", "Years In Role"]
].copy()

# Extract the starting calendar year from season strings
# Example: "2001-2002" -> 2001
gm["start_year"] = pd.to_numeric(
    gm["Start Season"].astype(str).str[:4],
    errors="coerce"
)

gm["end_year"] = pd.to_numeric(
    gm["End Season"].astype(str).str[:4],
    errors="coerce"
)

# Keep any tenure that overlaps the 2001-02 to 2025-26 window
gm = gm.loc[
    (gm["start_year"] <= 2025) &
    ((gm["end_year"] >= 2001) | gm["end_year"].isna())
].copy()

print("Rows remaining:", len(gm))
display(gm.head())

Rows remaining: 130


,Staff,Team,Start Season,End Season,Years In Role,start_year,end_year
1,Koby Altman,CLE,2017-2018,2021-2022,5,2017,2021.0
6,Pete Babcock,"DEN, ATL",1987-1988,2002-2003,16,1987,2002.0
7,Rob Babcock,TOR,2004-2005,2005-2006,2,2004,2005.0
9,Bob Bass,"SAN, NOP, NOP",1979-1980,2003-2004,22,1979,2003.0
10,Elgin Baylor,LAC,1986-1987,2007-2008,22,1986,2007.0


In [8]:
print("Original rows:", len(front_office))
print("Filtered rows:", len(gm))

print("Missing start years:", gm["start_year"].isna().sum())
print("Missing end years:", gm["end_year"].isna().sum())

display(gm.tail(10))

Original rows: 269
Filtered rows: 130
Missing start years: 0
Missing end years: 21


,Staff,Team,Start Season,End Season,Years In Role,start_year,end_year
245,Bobby Webster,TOR,2017-2018,-,10,2017,NaN
248,John Weisbrod,ORL,2004-2005,2004-2005,1,2004,2004.0
251,Jeff Weltman,TOR,2016-2017,2016-2017,1,2016,2016.0
252,Jerry West,"LAL, MEM",1982-1983,2006-2007,17,1982,2006.0
253,Bob Whitsitt,"OKC, POR",1986-1987,2002-2003,17,1986,2002.0
254,Wes Wilcox,ATL,2015-2016,2016-2017,2,2015,2016.0
262,Michael Winger,LAC,2018-2019,2022-2023,5,2018,2022.0
264,Dave Wohl,"MIA, LAC",1995-1996,2017-2018,6,1995,2017.0
265,Brian Wright,SAN,2019-2020,-,8,2019,NaN
267,Justin Zanik,UTH,2018-2019,-,9,2018,NaN


In [9]:
# Flag executives associated with multiple teams
gm["needs_review"] = gm["Team"].str.contains(",", na=False)

# Count flagged records
print("Total records:", len(gm))
print("Multi-team records:", gm["needs_review"].sum())

# Inspect the flagged records
display(
    gm.loc[
        gm["needs_review"],
        ["Staff", "Team", "Start Season", "End Season"]
    ]
)

Total records: 130
Multi-team records: 42


,Staff,Team,Start Season,End Season
6,Pete Babcock,"DEN, ATL",1987-1988,2002-2003
9,Bob Bass,"SAN, NOP, NOP",1979-1980,2003-2004
17,Bernie Bickerstaff,"DEN, CHA",1996-1997,2006-2007
23,Jeff Bower,"NOP, NOP, DET",2001-2002,2017-2018
27,Allan Bristow,"NOP, DEN, NOP",1990-1991,2004-2005
30,Chad Buchanan,"POR, IND",2011-2012,-
37,Rich Cho,"POR, CHA",2010-2011,2017-2018
40,Bryan Colangelo,"PHX, PHX, TOR",1995-1996,2012-2013
47,Mike D'Antoni,"DEN, PHX",1997-1998,2006-2007
62,"Mike Dunleavy, Sr.","MIL, LAC",1992-1993,2009-2010


In [10]:
import pandas as pd

# Start with one franchise to verify that pandas can access
# and interpret Basketball-Reference's executive history.
url = "https://www.basketball-reference.com/teams/PHO/executives.html"

tables = pd.read_html(url)

print("Tables found:", len(tables))

for i, table in enumerate(tables):
    print(f"Table {i}: {table.shape}")
    display(table.head())

Tables found: 1
Table 0: (9, 5)


,Rk,Executive,Start,End,Notes
0,1,Jerry Colangelo,1968,1994-02-23,NaN
1,2,Bryan Colangelo,1994-02-23,2006-02-27,NaN
2,3,Mike D'Antoni,2006-03-10,2007-06-06,NaN
3,4,Steve Kerr,2007-06-06,2010-06-30,NaN
4,5,Lance Blanks,2010-08-05,2013-04-22,NaN


In [11]:
table.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Rk         9 non-null      int64
 1   Executive  9 non-null      str  
 2   Start      9 non-null      str  
 3   End        9 non-null      str  
 4   Notes      1 non-null      str  
dtypes: int64(1), str(4)
memory usage: 492.0 bytes


In [12]:
# Current franchises; historical names will be handled separately.
teams = {
    "ATL": "Atlanta Hawks",
    "BOS": "Boston Celtics",
    "BRK": "Brooklyn Nets",
    "CHI": "Chicago Bulls",
    "CHO": "Charlotte Hornets",
    "CLE": "Cleveland Cavaliers",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "DET": "Detroit Pistons",
    "GSW": "Golden State Warriors",
    "HOU": "Houston Rockets",
    "IND": "Indiana Pacers",
    "LAC": "Los Angeles Clippers",
    "LAL": "Los Angeles Lakers",
    "MEM": "Memphis Grizzlies",
    "MIA": "Miami Heat",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "NOP": "New Orleans Pelicans",
    "NYK": "New York Knicks",
    "OKC": "Oklahoma City Thunder",
    "ORL": "Orlando Magic",
    "PHI": "Philadelphia 76ers",
    "PHO": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "SAC": "Sacramento Kings",
    "SAS": "San Antonio Spurs",
    "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz",
    "WAS": "Washington Wizards",
}

print("Franchises:", len(teams))

Franchises: 30


In [13]:
import pandas as pd
import time

# Store each franchise's executive table here.
all_executives = []

# Track any franchises whose pages fail to load.
failed_teams = []

for key, value in teams.items():

    # Construct the URL using the franchise code.
    url = f"https://www.basketball-reference.com/teams/{key}/executives.html"

    try:
        # Read the first HTML table (index 0).
        df = pd.read_html(url)[0]

        # Identify which franchise each executive belongs to.
        df["franchise_id"] = key
        df["franchise_name"] = value

        # Store this DataFrame for later combination.
        all_executives.append(df)

        print(f"SUCCESS: {key} — {len(df)} rows")

    except Exception as e:
        # Record errors without stopping the entire loop.
        failed_teams.append(key)
        print(f"FAILED: {key} — {e}")

    # Space out requests to avoid hammering the website.
    time.sleep(3)

print("\nSuccessful franchises:", len(all_executives))
print("Failed franchises:", failed_teams)

SUCCESS: ATL — 19 rows
SUCCESS: BOS — 9 rows
FAILED: BRK — `Import html5lib` failed.  Use pip or conda to install the html5lib package.
SUCCESS: CHI — 11 rows
FAILED: CHO — `Import html5lib` failed.  Use pip or conda to install the html5lib package.
SUCCESS: CLE — 11 rows
SUCCESS: DAL — 9 rows
SUCCESS: DEN — 18 rows
SUCCESS: DET — 23 rows
SUCCESS: GSW — 15 rows
SUCCESS: HOU — 9 rows
SUCCESS: IND — 10 rows
SUCCESS: LAC — 16 rows
SUCCESS: LAL — 12 rows
SUCCESS: MEM — 8 rows
SUCCESS: MIA — 4 rows
SUCCESS: MIL — 12 rows
SUCCESS: MIN — 13 rows
FAILED: NOP — `Import html5lib` failed.  Use pip or conda to install the html5lib package.
SUCCESS: NYK — 23 rows
SUCCESS: OKC — 13 rows
SUCCESS: ORL — 8 rows
SUCCESS: PHI — 26 rows
SUCCESS: PHO — 9 rows
SUCCESS: POR — 13 rows
SUCCESS: SAC — 14 rows
SUCCESS: SAS — 8 rows
SUCCESS: TOR — 8 rows
SUCCESS: UTA — 11 rows
SUCCESS: WAS — 14 rows

Successful franchises: 27
Failed franchises: ['BRK', 'CHO', 'NOP']


In [14]:
# Basketball-Reference executive-history URL overrides
executive_url_codes = {
    "BRK": "NJN",
    "CHO": "CHA",
    "NOP": "NOH"
}

# Retry only the previously failed franchises
still_failed = []

for key in failed_teams:

    name = teams[key]
    url_code = executive_url_codes[key]

    url = f"https://www.basketball-reference.com/teams/{url_code}/executives.html"

    try:
        df = pd.read_html(url)[0]

        # Preserve our standardized franchise identifier
        df["franchise_id"] = key
        df["franchise_name"] = name

        # Retain the original source identifier for traceability
        df["source_url_code"] = url_code

        all_executives.append(df)

        print(f"SUCCESS: {key} using {url_code} — {len(df)} rows")

    except Exception as e:
        still_failed.append(key)
        print(f"FAILED: {key} using {url_code} — {e}")

    time.sleep(3)

print("\nStill failed:", still_failed)

SUCCESS: BRK using NJN — 23 rows
SUCCESS: CHO using CHA — 10 rows
SUCCESS: NOP using NOH — 7 rows

Still failed: []


In [15]:
len(all_executives)

30

In [16]:
# Summarize all 30 DataFrames without concatenating them.
summary = pd.DataFrame([
    {
        "Franchise": df["franchise_id"].iloc[0],
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing_Start": df["Start"].isna().sum(),
        "Missing_End": df["End"].isna().sum()
    }
    for df in all_executives
])

display(summary)

,Franchise,Rows,Columns,Missing_Start,Missing_End
0,ATL,19,7,0,0
1,BOS,9,7,0,0
2,CHI,11,7,0,0
3,CLE,11,7,0,0
4,DAL,9,7,0,0
5,DEN,18,7,0,0
6,DET,23,7,0,0
7,GSW,15,7,0,0
8,HOU,9,7,0,0
9,IND,10,7,0,0


In [17]:
print("Franchises:", summary["Franchise"].nunique())
print("Total executive records:", summary["Rows"].sum())
print("Missing start values:", summary["Missing_Start"].sum())
print("Missing end values:", summary["Missing_End"].sum())

Franchises: 30
Total executive records: 386
Missing start values: 0
Missing end values: 0


In [18]:
# Combine the 30 individual franchise DataFrames into one.
# ignore_index=True creates a new continuous row index.
full_executive_list = pd.concat(all_executives, ignore_index=True)

# Three franchises used different URL codes.
# Fill missing source codes with the standard franchise ID.
full_executive_list["source_url_code"] = full_executive_list["source_url_code"].fillna(
    full_executive_list["franchise_id"]
)

# Validate the combined dataset.
print("Shape:", full_executive_list.shape)
print("Total rows:", len(full_executive_list))
print("Unique franchises:", full_executive_list["franchise_id"].nunique())

display(full_executive_list.head(10))
full_executive_list.info()

Shape: (386, 8)
Total rows: 386
Unique franchises: 30


,Rk,Executive,Start,End,Notes,franchise_id,franchise_name,source_url_code
0,1,Ben Kerner,1949,1960-11-26,NaN,ATL,Atlanta Hawks,ATL
1,2,Marty Blake,1960-11-26,1970-04-29,NaN,ATL,Atlanta Hawks,ATL
2,3,Bob Cousins,1970-04-29,1972-04-24,NaN,ATL,Atlanta Hawks,ATL
3,4,Richie Guerin,1972-04-24,1973-08-04,NaN,ATL,Atlanta Hawks,ATL
4,5,Pat Williams,1973-08-04,1974-08-06,NaN,ATL,Atlanta Hawks,ATL
5,6,William Wilcox,1974-08-06,1975-06-23,NaN,ATL,Atlanta Hawks,ATL
6,7,Bud Seretean,1975-06-23,1977-01-03,NaN,ATL,Atlanta Hawks,ATL
7,8,Mike Storen,1977-01-03,1977-09-27,NaN,ATL,Atlanta Hawks,ATL
8,9,Michael Gearon,1977-09-27,1979-07-12,NaN,ATL,Atlanta Hawks,ATL
9,10,Lewis Schaffel,1979-07-12,1979-11-19,NaN,ATL,Atlanta Hawks,ATL


<class 'pandas.DataFrame'>
RangeIndex: 386 entries, 0 to 385
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Rk               386 non-null    object
 1   Executive        386 non-null    str   
 2   Start            386 non-null    str   
 3   End              386 non-null    str   
 4   Notes            23 non-null     object
 5   franchise_id     386 non-null    str   
 6   franchise_name   386 non-null    str   
 7   source_url_code  386 non-null    str   
dtypes: object(2), str(6)
memory usage: 24.3+ KB


In [19]:
# Create a copy to preserve the complete source data.
gm_clean = full_executive_list.copy()

# Select only the columns needed for executive tenure analysis.
gm_clean = gm_clean[
    ["Executive", "Start", "End", "franchise_id", "franchise_name"]
].copy()

# Standardize column names.
gm_clean = gm_clean.rename(columns={
    "Executive": "executive_name",
    "Start": "start_date",
    "End": "end_date"
})

# Convert date strings into datetime objects.
# Unrecognized or missing dates become NaT.
gm_clean["start_date"] = pd.to_datetime(
    gm_clean["start_date"], errors="coerce"
)

gm_clean["end_date"] = pd.to_datetime(
    gm_clean["end_date"], errors="coerce"
)

# Inspect the cleaned structure.
print("Shape:", gm_clean.shape)
display(gm_clean.head())
gm_clean.info()

Shape: (386, 5)


,executive_name,start_date,end_date,franchise_id,franchise_name
0,Ben Kerner,1949-01-01,1960-11-26,ATL,Atlanta Hawks
1,Marty Blake,NaT,1970-04-29,ATL,Atlanta Hawks
2,Bob Cousins,NaT,1972-04-24,ATL,Atlanta Hawks
3,Richie Guerin,NaT,1973-08-04,ATL,Atlanta Hawks
4,Pat Williams,NaT,1974-08-06,ATL,Atlanta Hawks


<class 'pandas.DataFrame'>
RangeIndex: 386 entries, 0 to 385
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   executive_name  386 non-null    str           
 1   start_date      34 non-null     datetime64[us]
 2   end_date        346 non-null    datetime64[us]
 3   franchise_id    386 non-null    str           
 4   franchise_name  386 non-null    str           
dtypes: datetime64[us](2), str(3)
memory usage: 15.2 KB


In [20]:
print("Missing start dates:", gm_clean["start_date"].isna().sum())
print("Missing end dates:", gm_clean["end_date"].isna().sum())

display(
    gm_clean.loc[
        gm_clean["start_date"].isna() |
        gm_clean["end_date"].isna()
    ]
)

Missing start dates: 352
Missing end dates: 40


,executive_name,start_date,end_date,franchise_id,franchise_name
1,Marty Blake,NaT,1970-04-29,ATL,Atlanta Hawks
2,Bob Cousins,NaT,1972-04-24,ATL,Atlanta Hawks
3,Richie Guerin,NaT,1973-08-04,ATL,Atlanta Hawks
4,Pat Williams,NaT,1974-08-06,ATL,Atlanta Hawks
5,William Wilcox,NaT,1975-06-23,ATL,Atlanta Hawks
...,...,...,...,...,...
381,Jeff Bower,NaT,2010-07-13,NOP,New Orleans Pelicans
382,Dell Demps,NaT,2019-02-15,NOP,New Orleans Pelicans
383,Danny Ferry,NaT,2019-04-17,NOP,New Orleans Pelicans
384,David Griffin,NaT,2025-04-14,NOP,New Orleans Pelicans


In [21]:
# Inspect the original values, before datetime conversion.
display(
    full_executive_list[
        ["Executive", "Start", "End"]
    ].head(20)
)

# Count missing values in the original source columns.
print("Original missing starts:", full_executive_list["Start"].isna().sum())
print("Original missing ends:", full_executive_list["End"].isna().sum())

,Executive,Start,End
0,Ben Kerner,1949,1960-11-26
1,Marty Blake,1960-11-26,1970-04-29
2,Bob Cousins,1970-04-29,1972-04-24
3,Richie Guerin,1972-04-24,1973-08-04
4,Pat Williams,1973-08-04,1974-08-06
5,William Wilcox,1974-08-06,1975-06-23
6,Bud Seretean,1975-06-23,1977-01-03
7,Mike Storen,1977-01-03,1977-09-27
8,Michael Gearon,1977-09-27,1979-07-12
9,Lewis Schaffel,1979-07-12,1979-11-19


Original missing starts: 0
Original missing ends: 0


In [22]:
# Show original dates for rows that failed conversion.
failed_dates = full_executive_list.loc[
    gm_clean["start_date"].isna(),
    ["Executive", "Start", "End"]
]

display(failed_dates.head(15))

,Executive,Start,End
1,Marty Blake,1960-11-26,1970-04-29
2,Bob Cousins,1970-04-29,1972-04-24
3,Richie Guerin,1972-04-24,1973-08-04
4,Pat Williams,1973-08-04,1974-08-06
5,William Wilcox,1974-08-06,1975-06-23
6,Bud Seretean,1975-06-23,1977-01-03
7,Mike Storen,1977-01-03,1977-09-27
8,Michael Gearon,1977-09-27,1979-07-12
9,Lewis Schaffel,1979-07-12,1979-11-19
10,Stan Kasten,1979-11-19,1990-02-14


In [23]:
# Allow pandas to recognize different date formats in the same column.
gm_clean["start_date"] = pd.to_datetime(
    full_executive_list["Start"],
    format="mixed",
    errors="coerce"
)

gm_clean["end_date"] = pd.to_datetime(
    full_executive_list["End"],
    format="mixed",
    errors="coerce"
)

print("Missing starts:", gm_clean["start_date"].isna().sum())
print("Missing ends:", gm_clean["end_date"].isna().sum())

Missing starts: 4
Missing ends: 34


In [24]:
# Research window: 2001-02 through 2025-26 NBA seasons
study_start = pd.Timestamp("2001-07-01")
study_end = pd.Timestamp("2026-06-30")

# Condition 1: Executive started before the research period ended.
started_in_time = gm_clean["start_date"] <= study_end

# Condition 2: Executive left after the research period began.
ended_in_time = gm_clean["end_date"] >= study_start

# Condition 3: Executive is still serving ("Present" in source).
still_active = (
    full_executive_list["End"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("present")
    .fillna(False)
)

# Keep tenures that overlap our research window.
overlap_mask = started_in_time & (ended_in_time | still_active)

# Apply the mask to select qualifying executive tenures.
gm_filtered = gm_clean.loc[overlap_mask].copy()


In [25]:
print("Original records:", len(gm_clean))
print("Filtered records:", len(gm_filtered))
print("Records removed:", len(gm_clean) - len(gm_filtered))

print("Unique franchises:", gm_filtered["franchise_id"].nunique())

# Confirm that no retained tenure falls entirely outside the window.
print(
    "Invalid retained records:",
    (
        (gm_filtered["start_date"] > study_end) |
        (gm_filtered["end_date"] < study_start)
    ).sum()
)

display(gm_filtered.head(10))
display(gm_filtered.tail(10))

Original records: 386
Filtered records: 181
Records removed: 205
Unique franchises: 30
Invalid retained records: 0


,executive_name,start_date,end_date,franchise_id,franchise_name
11,Pete Babcock,1990-02-14,2003-04-02,ATL,Atlanta Hawks
12,Billy Knight,2003-04-02,2008-05-28,ATL,Atlanta Hawks
13,Rick Sund,2008-05-28,2012-06-25,ATL,Atlanta Hawks
14,Danny Ferry,2012-06-25,2014-09-12,ATL,Atlanta Hawks
15,Mike Budenholzer,2014-09-12,2017-05-25,ATL,Atlanta Hawks
16,Travis Schlenk,2017-05-25,2022-12-21,ATL,Atlanta Hawks
17,Landry Fields,2022-12-21,2025-04-21,ATL,Atlanta Hawks
18,Onsi Saleh,2025-04-21,NaT,ATL,Atlanta Hawks
25,Chris Wallace,2001-01-08,2003-05-09,BOS,Boston Celtics
26,Danny Ainge,2003-05-09,2021-06-02,BOS,Boston Celtics


,executive_name,start_date,end_date,franchise_id,franchise_name
376,Rich Cho,2011-06-14,2018-04-08,CHO,Charlotte Hornets
377,Mitch Kupchak,2018-04-08,2024-02-12,CHO,Charlotte Hornets
378,Jeff Peterson,2024-03-05,NaT,CHO,Charlotte Hornets
379,Bob Bass,2002-05-16,2004-06-17,NOP,New Orleans Pelicans
380,Allan Bristow,2004-06-17,2005-10-01,NOP,New Orleans Pelicans
381,Jeff Bower,2005-10-01,2010-07-13,NOP,New Orleans Pelicans
382,Dell Demps,2010-07-21,2019-02-15,NOP,New Orleans Pelicans
383,Danny Ferry,2019-02-15,2019-04-17,NOP,New Orleans Pelicans
384,David Griffin,2019-04-17,2025-04-14,NOP,New Orleans Pelicans
385,Joe Dumars,2025-04-17,NaT,NOP,New Orleans Pelicans


In [26]:
display(
    full_executive_list.loc[
        gm_clean["start_date"].isna(),
        ["Executive", "Start", "End", "franchise_id"]
    ]
)

,Executive,Start,End,franchise_id
97,Executive,Start,End,DET
219,Executive,Start,End,NYK
263,Executive,Start,End,PHI
366,Executive,Start,End,BRK


In [27]:
# Check for missing executive names or start dates.
print("Missing executives:", gm_filtered["executive_name"].isna().sum())
print("Missing start dates:", gm_filtered["start_date"].isna().sum())

# Check for exact duplicate executive-tenure records.
print("Duplicate records:", gm_filtered.duplicated().sum())

# Verify that all 30 franchises survived filtering.
print("Franchises:", gm_filtered["franchise_id"].nunique())

# Check that every retained record meets our overlap rule.
print("Invalid starts:", (gm_filtered["start_date"] > study_end).sum())
print("Invalid ends:", (gm_filtered["end_date"] < study_start).sum())

Missing executives: 0
Missing start dates: 0
Duplicate records: 0
Franchises: 30
Invalid starts: 0
Invalid ends: 0


In [28]:
from pathlib import Path

# Define the staging directory relative to the notebooks folder.
staging_dir = Path("../data/staging/front_office")
staging_dir.mkdir(parents=True, exist_ok=True)

# Save the cleaned and filtered executive-tenure dataset.
output_path = staging_dir / "S15_GM_Tenures.csv"

gm_filtered.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(gm_filtered))

Saved: ../data/staging/front_office/S15_GM_Tenures.csv
Rows: 181


In [31]:
sanity_check = pd.read_csv("../data/staging/front_office/S15_GM_Tenures.csv")

sanity_check.shape

(181, 5)